# Gold Set 생성 — Step 1: Retriever 후보 풀 수집

시나리오별로 BM25 / Dense / Hybrid 검색을 실행하고, 결과를 머지하여 후보 도서 풀을 저장한다.

---

**입력**
- `evaluation/dataset/scenario_data.json` — 21개 평가 시나리오

**출력**
- `evaluation/dataset/goldset_candidates.jsonl` — 시나리오별 후보 도서 풀
  - 한 줄 = 한 도서 후보 (`query_id` + 도서 메타데이터 + `retrieval_info`)

**다음 단계**
- `01b_goldset_llm_judge.ipynb` — 후보 풀에 LLM judge relevance grade 부여

In [1]:
import os
os.chdir("/home/ubuntu/work/dahyeon/backend")
os.getcwd()

'/home/ubuntu/work/dahyeon/backend'

In [2]:
from app.modules.RAG.retriever import (
    full_bm25,
    full_dense,
    full_hybrid,
    chunk_dense,
    chunk_hybrid
)
from app.modules.RAG.anchor_book_pipeline import run_anchor_pipeline
import json
from pathlib import Path
from tqdm import tqdm
from copy import deepcopy
import warnings
warnings.filterwarnings('ignore')

/home/ubuntu/anaconda3/envs/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ubuntu/anaconda3/envs/venv/lib/python3.11/site-packages/elasticsearch/_sync/client/__init__.py:404: SecurityWarning: Connecting to 'https://localhost:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(


In [3]:
REPO_ROOT     = Path("/home/ubuntu/work/dahyeon")
DATASET_DIR   = REPO_ROOT / "evaluation" / "dataset"
SCENARIO_PATH = DATASET_DIR / "scenario_data.json"

with open(SCENARIO_PATH, encoding="utf-8") as f:
    scenarios = json.load(f)

print(f"시나리오 수: {len(scenarios)}")

REWRITTEN_SCENARIO_PATH = DATASET_DIR / "scenario_data_after_anchor_rewrite.json"

updated_scenarios = deepcopy(scenarios)

def extract_rag_query(scenario: dict) -> dict:
    """마지막 turn의 rag_query를 꺼내고 query_id(=scenario_id)를 추가한다."""
    rag_query = scenario["turns"][-1]["rag_query"].copy()
    rag_query["query_id"] = scenario["scenario_id"]
    return rag_query

시나리오 수: 21


In [4]:
def simplify_item(item):
    return {
        "isbn":         item.get("isbn"),
        "title":        item.get("title"),
        "author":       item.get("author"),
        "publisher":    item.get("publisher"),
        "publish_date": item.get("publish_date"),
        "page":         item.get("page"),
        "large_cate":   item.get("large_cate"),
        "mid_cate":     item.get("mid_cate"),
        "small_cate":   item.get("small_cate"),
        "book_intro":   item.get("book_intro"),
        "book_index":   item.get("book_index_text"),
        "review_count": item.get("review_count"),
        "review_text":  item.get("review_text"),
    }


def merge_with_sources(result_groups: dict, key: str = "isbn") -> list:
    """여러 retriever 결과를 ISBN 기준으로 머지하고, retrieval_info를 누적한다."""
    merged = {}

    for source_name, results in result_groups.items():
        for item in results:
            item_key = item.get(key)
            if item_key is None:
                continue

            if item_key not in merged:
                merged[item_key] = simplify_item(item)
                merged[item_key]["retrieval_info"] = []

            merged[item_key]["retrieval_info"].append({
                "source": source_name,
                "rank":   item.get("rank"),
                "score":  item.get("score"),
            })

    return list(merged.values())

In [11]:
CANDIDATES_PATH = DATASET_DIR / "goldset_candidates.jsonl"
CANDIDATES_PATH.parent.mkdir(parents=True, exist_ok=True)

TOP_K = 40

RETRIEVAL_CONFIGS = {
    "bm25_full": {
        "func": full_bm25,
        "index_name": "books_review_full",
        "group": "bm25",
    },

    "dense_strategy0": {
        "func": full_dense,
        "index_name": "books_review_full",
        "group": "dense",
    },
    "dense_strategy1": {
        "func": chunk_dense,
        "index_name": "books_chunk_strategy1",
        "group": "dense",
    },
    "dense_strategy2": {
        "func": full_dense,
        "index_name": "books_chunk_strategy2",
        "group": "dense",
    },

    "hybrid_strategy0": {
        "func": full_hybrid,
        "index_name": "books_review_full",
        "group": "hybrid",
    },
    "hybrid_strategy1": {
        "func": chunk_hybrid,
        "index_name": "books_chunk_strategy1",
        "group": "hybrid",
    },
    "hybrid_strategy2": {
        "func": full_hybrid,
        "index_name": "books_chunk_strategy2",
        "group": "hybrid",
    },
}

processed_query_ids = set()

if CANDIDATES_PATH.exists():
    with CANDIDATES_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                item = json.loads(line)
                if item.get("query_id"):
                    processed_query_ids.add(item["query_id"])
            except json.JSONDecodeError:
                continue

print(f"이미 처리된 시나리오: {sorted(processed_query_ids)}")

scenario_stats = []

with CANDIDATES_PATH.open("a", encoding="utf-8") as f:

    for idx, scenario in enumerate(tqdm(updated_scenarios, desc="시나리오 검색")):

        rag_query = extract_rag_query(scenario)
        query_id = rag_query["query_id"]

        if query_id in processed_query_ids:
            print(f"skip: {query_id}")
            continue

        if rag_query.get("anchor"):
            rewritten_rag_query = run_anchor_pipeline(rag_query)

            kw = rewritten_rag_query.get("keyword_query") or []
            sq = rewritten_rag_query.get("semantic_query") or ""

            print(f"\n[{query_id}]")
            print(f"  keyword_query : {kw}")
            print(f"  semantic_query: {sq}")

            rewritten_rag_query["keyword_query"] = kw
            rewritten_rag_query["semantic_query"] = sq
            rewritten_rag_query["query_id"] = query_id

            scenario["turns"][-1]["rag_query"] = {
                k: v for k, v in rewritten_rag_query.items()
                if k != "query_id"
            }

            rag_query = rewritten_rag_query

        result_groups = {}

        for source_name, config in RETRIEVAL_CONFIGS.items():
            retrieval_func = config["func"]
            index_name = config["index_name"]
            retrieval_group = config["group"]

            print(f"  → {source_name}")

            kwargs = {
                "size": TOP_K,
                "index_name": index_name,
                "small_category_embeddings": None,
            }

            if retrieval_group in ["dense", "hybrid"]:
                kwargs["embedding_field"] = "embedding_kure"
                kwargs["embedding_model"] = "kure"

            results = retrieval_func(
                rag_query,
                **kwargs
            )

            result_groups[source_name] = results

        merged_candidates = merge_with_sources(result_groups)

        for book in merged_candidates:
            save_item = {
                **book,
                "query_id": query_id,
                "rag_query": rag_query,
            }

            f.write(json.dumps(save_item, ensure_ascii=False) + "\n")

        f.flush()

        processed_query_ids.add(query_id)

        scenario_stats.append({
            "query_id": query_id,
            "candidate_count": len(merged_candidates)
        })

        print(f"  → 중복 제거 후 {len(merged_candidates)}개 후보 저장")

print(f"\n저장 완료: {CANDIDATES_PATH}")

이미 처리된 시나리오: []


시나리오 검색:   0%|          | 0/21 [00:00<?, ?it/s]


[S01]
  keyword_query : ['실존주의적 주제', '철학적 성찰', '인문학 에세이', '심오한 통찰', '존재론적 질문', '삶의 의미']
  semantic_query: 실존주의적 주제를 탐구하며 철학적 성찰을 통해 독자에게 깊은 사색을 유도하는 인문학 에세이
  → bm25_full
  → dense_strategy0
  → dense_strategy1
  → dense_strategy2
  → hybrid_strategy0
  → hybrid_strategy1
  → hybrid_strategy2


시나리오 검색:   5%|▍         | 1/21 [00:24<08:15, 24.76s/it]

  → 중복 제거 후 115개 후보 저장

[S02]
  keyword_query : ['한국 경제', '문제점', '해결 방안', '경제/경영', '신자유주의 비판']
  semantic_query: 한국 경제의 문제점과 현실적인 해결 방안을 제시하는 경제/경영 분야의 책
  → bm25_full
  → dense_strategy0
  → dense_strategy1
  → dense_strategy2
  → hybrid_strategy0
  → hybrid_strategy1
  → hybrid_strategy2


...

  → 중복 제거 후 136개 후보 저장
  → bm25_full
  → dense_strategy0
  → dense_strategy1
  → dense_strategy2
  → hybrid_strategy0
  → hybrid_strategy1
  → hybrid_strategy2


시나리오 검색:  95%|█████████▌| 20/21 [03:46<00:08,  8.27s/it]

  → 중복 제거 후 134개 후보 저장
  → bm25_full
  → dense_strategy0
  → dense_strategy1
  → dense_strategy2
  → hybrid_strategy0
  → hybrid_strategy1
  → hybrid_strategy2


시나리오 검색: 100%|██████████| 21/21 [03:49<00:00, 10.95s/it]

  → 중복 제거 후 131개 후보 저장

저장 완료: /home/ubuntu/work/dahyeon/evaluation/dataset/goldset_candidates.jsonl


In [14]:
import json
from pathlib import Path

# =====================================================
# 파일 경로
# =====================================================

JSONL_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/goldset_candidates.jsonl")
JSON_PATH  = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/goldset_candidates.json")

# =====================================================
# jsonl -> json
# =====================================================

items = []

with JSONL_PATH.open("r", encoding="utf-8") as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        items.append(json.loads(line))

# =====================================================
# 저장
# =====================================================

with JSON_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        items,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"저장 완료: {JSON_PATH}")
print(f"총 개수: {len(items):,}")

저장 완료: /home/ubuntu/work/dahyeon/evaluation/dataset/goldset_candidates.json
총 개수: 2,543


In [13]:
# 전체 통계
from collections import defaultdict

query_counts = defaultdict(int)

with CANDIDATES_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        item = json.loads(line)
        query_counts[item.get("query_id", "unknown")] += 1

total = sum(query_counts.values())
print(f"{'query_id':>10}  {'후보 수':>8}")
print("-" * 22)
for qid in sorted(query_counts):
    print(f"{qid:>10}  {query_counts[qid]:>8}")
print("-" * 22)
print(f"{'합계':>10}  {total:>8}")

  query_id      후보 수
----------------------
       S01       115
       S02       115
       S03        85
       S04       102
       S05       130
       S06       120
       S07        67
       S08       125
       S09       137
       S10       116
       S11       141
       S12       125
       S13       137
       S14       130
       S15       114
       S16       145
       S17       116
       S18       122
       S19       136
       S20       134
       S21       131
----------------------
        합계      2543
